In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:

# ==========================================
# 1. CLIENT CLASS (Local Training)
# ==========================================
class FederatedClient:
    def __init__(self, client_id, X, y):
        self.client_id = client_id
        self.X = X
        self.y = y
        # Local model: Stochastic Gradient Descent for Linear Regression
        self.model = SGDRegressor(max_iter=5, tol=1e-3, learning_rate='constant', eta0=0.01)

    def local_update(self, global_weights, global_intercept):
        """
        Receives global parameters, trains locally, and returns updated parameters.
        """
        # Synchronize with the global model parameters
        if global_weights is not None:
            self.model.coef_ = np.copy(global_weights)
            self.model.intercept_ = np.copy(global_intercept)
        
        # Perform local training (1 epoch / partial fit)
        self.model.partial_fit(self.X, self.y)
        
        # Return local weights, intercept, and the number of samples (for weighting)
        return self.model.coef_, self.model.intercept_, len(self.X)



In [3]:
# ==========================================
# 2. SERVER CLASS (Aggregation & Distribution)
# ==========================================
class FederatedServer:
    def __init__(self, input_dim):
        # Initialize global model parameters
        self.global_weights = np.zeros(input_dim)
        self.global_intercept = np.array([0.0])

    def aggregate(self, client_updates):
        """
        Performs Weighted Federated Averaging (FedAvg).
        """
        total_samples = sum(update[2] for update in client_updates)
        
        new_weights = np.zeros_like(self.global_weights)
        new_intercept = np.zeros_like(self.global_intercept)
        
        for coef, intercept, n_k in client_updates:
            # weight = (samples in client k) / (total samples across all clients)
            weight_factor = n_k / total_samples
            new_weights += coef * weight_factor
            new_intercept += intercept * weight_factor
            
        self.global_weights = new_weights
        self.global_intercept = new_intercept

    def evaluate(self, X_test, y_test):
        """
        Evaluates the global model on a held-out test set.
        """
        predictions = np.dot(X_test, self.global_weights) + self.global_intercept
        mse = mean_squared_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        return mse, r2



In [4]:
# ==========================================
# 3. MAIN SIMULATION PIPELINE
# ==========================================
def run_federated_learning(num_clients=5, rounds=20):
    # --- Data Preparation ---
    housing = fetch_california_housing()
    X, y = housing.data, housing.target
    
    # Standardize features for better convergence
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # Split into Train (for clients) and Test (for server evaluation)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Partition training data across clients (IID Sharding)
    X_shards = np.array_split(X_train, num_clients)
    y_shards = np.array_split(y_train, num_clients)
    
    # --- Initialization ---
    server = FederatedServer(input_dim=X.shape[1])
    clients = [FederatedClient(i, X_shards[i], y_shards[i]) for i in range(num_clients)]
    
    print(f"Starting Training: {num_clients} clients, {rounds} rounds.\n")
    print(f"{'Round':<10} | {'MSE (Global)':<15} | {'R2 Score':<10}")
    print("-" * 45)

    # --- Federated Learning Loop ---
    for r in range(rounds):
        local_updates = []
        
        # Phase 1: Distribution & Local Training
        for client in clients:
            update = client.local_update(server.global_weights, server.global_intercept)
            local_updates.append(update)
            
        # Phase 2: Server Aggregation (FedAvg)
        server.aggregate(local_updates)
        
        # Phase 3: Global Evaluation
        if r % 2 == 0 or r == rounds - 1:
            mse, r2 = server.evaluate(X_test, y_test)
            print(f"{r:<10} | {mse:<15.4f} | {r2:<10.4f}")

    print("\nTraining Complete.")
    print(f"Final Global Weights: {server.global_weights.round(3)}")

if __name__ == "__main__":
    run_federated_learning(num_clients=8, rounds=30)

Starting Training: 8 clients, 30 rounds.

Round      | MSE (Global)    | R2 Score  
---------------------------------------------
0          | 997.1235        | -759.9255 
2          | 26799508165524.3359 | -20451254997885.3438
4          | 268583330637092945920.0000 | -204961454856361508864.0000
6          | 169883812058792787968.0000 | -129641825475620421632.0000
8          | 46394920537659899904.0000 | -35404916562721280000.0000
10         | 308339643649282998272.0000 | -235300313695347113984.0000
12         | 601688031311018459136.0000 | -459160492107378393088.0000
14         | 157128432970025304064.0000 | -119907933766567608320.0000
16         | 333765664534815899648.0000 | -254703432345880920064.0000
18         | 118298721135276048384.0000 | -90276183313457545216.0000
20         | 305434501175285710848.0000 | -233083339817545138176.0000
22         | 269283648505194250240.0000 | -205495881802246291456.0000
24         | 261407872222290083840.0000 | -199485715194965688320.0000
26   